In [ ]:
import torch
import torch.nn as nn

from dscribe.descriptors import SOAP
import ase

from ase import Atoms

import matplotlib.pyplot as plt

## Atom Embedding

In [ ]:
class PowerSpectrumAtomEmbedding(nn.Module):
    """
    Embedding of the atoms using the power spectrum.
    """
    def __init__(self, species: list, r_cut: float, n_max: int, l_max: int):
        """
        Constructor of the node embedding.
        
        Parameters
        ----------
        dimension : int
            Dimension of the embedding.
        """
        super().__init__()
                
        self.soap = SOAP(
            species=species,
            r_cut=r_cut,
            n_max=n_max,
            l_max=l_max,
            sigma=0.3,
            periodic=False,
        )
        
    def forward(self, x):
        """
        Call method of the embedding.
        
        Parameters
        ----------
        x : torch.Tensor
            Input to the neural network.
        """
        return torch.tensor(self.soap.create(x), dtype=torch.float32)

## Node embedding

In [ ]:
class AttentionHead(nn.Module):
    """
    Attention head for the nodes or edges.
    """
    def __init__(self, atom_dimension: int, embedding_dimension: int):
        """
        Constructor for the attention head.
        """
        super().__init__()
        self.q_weights = nn.Linear(atom_dimension, embedding_dimension)
        self.k_weights = nn.Linear(atom_dimension, embedding_dimension)
        self.v_weights = nn.Linear(atom_dimension, embedding_dimension)

        self.norm = torch.sqrt(embedding_dimension)

    def forward(self, x):
        """
        Call method for the attention head.
        
        Parameters
        ----------
        x : torch.Tensor
            Input to the neural network.
        """
        q = self.q_weights(x)
        k = self.k_weights(x)
        v = self.v_weights(x)

        attention = nn.functional.softmax(torch.matmul(q, k.transpose(-2,-1)), dim=-1) # / self.norm
        
        return torch.matmul(attention, v)

In [ ]:
class AttentionBlock(nn.Module):
    """
    Attention block for the nodes or edges. 
    """
    def __init__(self, atom_dimension: int, embedding_dimension: int, num_heads: int):
        """
        Constructor for the attention block.
        """
        super().__init__()
        self.heads = nn.ModuleList(
            [AttentionHead(atom_dimension, embedding_dimension) for _ in range(num_heads)]
        )

        self.linear = nn.Linear(embedding_dimension * num_heads, embedding_dimension)

    def forward(self, x):
        """
        Call method for the attention block.
        
        Parameters
        ----------
        x : torch.Tensor
            Input to the neural network.
        """
        
        return self.linear(torch.cat([head(x) for head in self.heads], dim=-1))

In [ ]:
class NodeEmbedding(nn.Module):
    """
    Embedding of the nodes.
    """
    def __init__(
            self, 
            monomer_atoms: int, 
            atom_embedding: PowerSpectrumAtomEmbedding,
            in_features: int,
            out_features: int
        ):
        """
        Constructor of the node embedding.
        
        Parameters
        ----------
        dimension : int
            Dimension of the embedding.
        """
        super().__init__()
                
        self.atom_embedding = atom_embedding
        self.monomer_atoms = monomer_atoms

        self.input_block = AttentionBlock(
            in_features, torch.tensor(1080), 3
        )

        self.blocks = [
            AttentionBlock(
                torch.tensor(1080), torch.tensor(1080), 3
            ),
            AttentionBlock(
                torch.tensor(1080), torch.tensor(1080), 3
            ),
        ]

        self.linear = nn.Linear(1080, out_features)

    def atomic_output(self, x):
        """
        Compute an n atoms output.

        Output
        ------
        (n_atoms, dimension)
        """
        pass

    def monomer_output(self, x):
        """
        Compute a monomer output. 

        Output
        ------
        (dimension,)
        """
        pass
        
    def forward(self, x):
        """
        Call method of the embedding.
        
        Parameters
        ----------
        x : torch.Tensor
            Input to the neural network.
        """
        x = self.atom_embedding(x)
        x = nn.functional.relu(self.input_block(x))

        for block in self.blocks:
            x = block(x)
            x = nn.functional.relu(x)

        return self.linear(x)
        

In [ ]:
class NodeDecoder(nn.Module):
    """
    Embedding of the nodes.
    """
    def __init__(
            self, 
            monomer_atoms: int, 
            in_features: int,
            out_features: int
        ):
        """
        Constructor of the node embedding.
        
        Parameters
        ----------
        dimension : int
            Dimension of the embedding.
        """
        super().__init__()
                
        self.monomer_atoms = monomer_atoms

        self.input_block = AttentionBlock(
            in_features, torch.tensor(1080), 3
        )

        self.blocks = [
            AttentionBlock(
                torch.tensor(1080), torch.tensor(1080), 3
            ),
            AttentionBlock(
                torch.tensor(1080), torch.tensor(1080), 3
            ),
        ]
        self.linear = nn.Linear(1080, out_features)

    def atomic_output(self, x):
        """
        Compute an n atoms output.

        Output
        ------
        (n_atoms, dimension)
        """
        pass

    def monomer_output(self, x):
        """
        Compute a monomer output. 

        Output
        ------
        (dimension,)
        """
        pass
        
    def forward(self, x):
        """
        Call method of the embedding.
        
        Parameters
        ----------
        x : torch.Tensor
            Input to the neural network.
        """
        x = nn.functional.relu(self.input_block(x))
        for block in self.blocks:
            x = block(x)
            x = nn.functional.relu(x)
            
        return self.linear(x)
        

## Methane Test

In [ ]:
# Define the positions
d = 1.09  # approximate C-H bond length in Angstroms
positions = [
    (0, 0, 0),  # Carbon atom
    (d, 0, 0),  # Hydrogen atom
    (-d, 0, 0), # Hydrogen atom
    (0, d, 0),  # Hydrogen atom
    (0, 0, d)   # Hydrogen atom
]

# Create methane molecule
methane = Atoms('CH4', positions=positions)

In [ ]:
atom_embedding = PowerSpectrumAtomEmbedding(
    species=["H", "C"],
    r_cut=5.0,
    n_max=8,
    l_max=9,
)

In [ ]:
encoder = NodeEmbedding(
    monomer_atoms=torch.tensor(5), 
    atom_embedding=atom_embedding,
    in_features=atom_embedding.soap.get_number_of_features(),
    out_features=torch.tensor(1048)
)

In [ ]:
decoder = NodeDecoder(
    monomer_atoms=torch.tensor(5), 
    in_features=1048,
    out_features=3
)

In [ ]:
enc_opt = torch.optim.Adam(encoder.parameters(), lr=0.001)
dec_opt = torch.optim.Adam(decoder.parameters(), lr=0.001)

In [ ]:
losses = []
n_epochs = 50 


for _ in range(n_epochs):
    enc_opt.zero_grad()
    dec_opt.zero_grad()
    representation = encoder(methane)
    reconstruction = decoder(representation)
    loss = torch.nn.functional.mse_loss(reconstruction, torch.tensor(methane.get_positions(), dtype=torch.float32))
    losses.append(loss.detach().numpy())
    loss.backward()

    enc_opt.step()
    dec_opt.step()

In [ ]:
plt.plot(losses)
plt.yscale("log")

In [ ]:
reconstruction

In [ ]:
methane.get_positions()